# Game Engine

In this notebook, we're going to complete the objective for week three: **building the engine**. The engine will be based on our flagship version of an LLM—Gemini 3 Flash—which was prompted using few-shot, chain of thought (CoT), and versioning. Additionally, we'll be supporting input and output (I/O) from the user. This means:

* Piping the relevant information from our latest few-shot file
* Creating an architecture that incorporates our setlist of skits
* Defining the logic behind the branching (if any)
* Toying with other prompt practices to make NPC interactions better

In [ ]:
# Install the official Google Generative AI SDK ('-q' silences the output, '-U' ensures the most up-to-date version)
!pip install -q -U google-generativeai

In [ ]:
# Add the dependencies (may need more/less)
import textwrap # new line
import re
import time
from google import genai
from google.genai import types
from google.colab import userdata

# Fetch API key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key = GEMINI_API_KEY)

In [ ]:
# Interpolate the few-shot-prompted LLM to build the evaluator
evaluator_instruction = (
    "You are a professional comedy writer. Evaluate whether the text is funny or not based on this binary rubric:\n"
    "0 (not funny) if there's no comedic intent, missing a punchline, feels forced, too niche, or unoriginal.\n"
    "1 (funny) if it's unpredictable yet clever, witty and unique, intuitive in its wordplay, or isn't overty offensive.\n"
    "Please be lenient and make it easy for responses to obtain a 1. Assume that delivery, timing, and commitment are all in the user's favor.\n"
    "First, write one brief sentence explaining your reasoning. Then, on a new line, output the final score in this exact format: 'Score: [0/1]'."
)

few_shot_examples = [
    # Hard-coded three examples of class 0 & reasons for their classification
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: I used to date Hispanic guys, but now I prefer consensual.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is a racist rape joke mocking Hispanic men, making it both overly crude and insensitive. \nScore: 0")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: You can get a lot of power out of one inch.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is low and almost too easy of a joke to make, so it's unoriginal and not funny. \nScore: 0")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: No one notices the Taylor oil spill because it's a disaster taking place over a long period of time, like Derrick Rose's career.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This comes off as antagonistic since it throws shade at a person, Derrick Rose (whom many consider to be a great but flawed NBA player), and is too unpopular of an opinion, thus not constituting a good joke. \nScore: 0")]),

    # Hard-coded three examples of class 1 & reasons for their classification
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Please, don't call me sir. Call me ma'am.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because it is witty and self-deprecating, given that the user is indeed a man. \nScore: 1")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: Hey, I am Conan O'Brien and I'm honored to be the last human host of the Academy Awards. Yes! Yeah! Next year, it's going to be a Waymo in a tux.")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is very funny as it's clever and uses observational comedy to comment on the advent of AI in self-driving cars, like Waymo, whilst keeping it wholesome. \nScore: 1")]),
    types.Content(role = 'user', parts = [types.Part.from_text(text = "Text: How did us teaching Diana to drive turn into I'm a male prostitute, you're going to put me out, and you're going to come back in an hour, and you want your trap full?")]),
    types.Content(role = 'model', parts = [types.Part.from_text(text = "Reasoning: This is funny because, in spite of its unorthodox format (as it doesn't read like a classic joke or pun), it calls out the absurdity of the situation in a unique and unpredictable way. \nScore: 1")])
]

def get_humor_score(player_text):
    turn_input = types.Content(role = 'user', parts = [types.Part.from_text(text = f"Text: {player_text}")])
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = few_shot_examples + [turn_input],
        config = types.GenerateContentConfig(
            system_instruction = evaluator_instruction,
            temperature = 0
        )
    )
    match = re.search(r'Score:\s*([0-1])', response.text)
    return int(match.group(1)) if match else 0

In [ ]:
# Add a global variable defining the instructions for the game / dungeon master
INSTRUCTION = (
    "You are the (dungeon) master for an interactive text-based game in the style of 'Choice of Games.'\n"
    "Your job is to take a basic outline of a skit and expand it into an immersive RPG scene using the following pointers:\n"
    "1. Narrate in the second person (e.g. 'You pull up to...' or 'Your heart races.')\n"
    "2. Write one to two paragraphs totalling no more than half a dozen sentences. Let the complexity of the outline dictate the exact length, just like a real novel\n"
    "3. Ground the player in the scene by describing the setting (i.e. one or more of the five senses—sight, smell, touch, taste, hearing)\n"
    "4. Include at least one piece of dialogue from an NPC interwoven into the scene\n"
    "5. End the final paragraph with an NPC interacting directly to the player. Leave the narrative hanging so that the player must respond."
)

In [ ]:
# Create the game / dungeon master whose job is to add to & connect the skits
def cold_open():
    prompt = "Write an abridged SNL (Saturday Night Live)-style cold open for a text-based comedy role-playing game. End with the phrase, 'Live from BU, it's The Comedy Game!'"
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            temperature = 0.5
        )
    )
    return response.text

def skitzo(segment_text):
    prompt = f"Here's the outline to expand upon: {segment_text}"
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            system_instruction = INSTRUCTION,
            temperature = 0.5
        )
    )
    return response.text

def bit(previous_scene, player_action, npc_reaction, bit_text):
    prompt = (
        f"Earlier in this scene: {previous_scene}\n"
        f"Player said: {player_action}\n"
        f"NPC reacted: {npc_reaction}\n\n"
        f"Now, seamlessly continue the narrative based on this next plot point: {bit_text}"
    )
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            system_instruction = INSTRUCTION,
            temperature = 0.5
        )
    )
    return response.text

def npc(scenario, player_text, score, wrap = False):
    reaction_type = "positive, laughing, and rewarding" if score == 1 else "awkward, offended, or confused"
    WRAP_INSTRUCTION = INSTRUCTION
    if wrap:
        WRAP_INSTRUCTION = INSTRUCTION.replace(
            "5. End the final paragraph with an NPC interacting directly to the player. Leave the narrative hanging so that the player must respond.",
            "5. End the final paragraph by wrapping up the bit and segment."
        )
    prompt = (
        f"Context: {scenario}\n"
        f"Player's response: {player_text}\n\n"
        f"The player's response was evaluated for humor and got a score of {score}/1. "
        f"Write a short, {reaction_type} response from the NPC reacting to the player."
    )
    response = client.models.generate_content(
        model = "gemini-3-flash-preview",
        contents = prompt,
        config = types.GenerateContentConfig(
            system_instruction = INSTRUCTION,
            temperature = 0.5
        )
    )
    return response.text

In [ ]:
# Initialize the skits
skits = [
    {
        "Segment": "The user is a college exchange student who wants to sound like a native, so they take a crash course in a foreign language—with special emphasis on learning slang—from an instructor and dialect coach. It is provided that the LLM gives translations for the other tongue in parentheses (akin to subtitles) so that the user may respond in English.",
        "Bit": "The user tries to learn how to say things like their name, greetings, phrases, questions, and body parts."
    },
    {
        "Segment": "The user crashes a club meeting they have never been to before. Is it for freebies? Is it for…um, kicks and giggles? Maybe getting a guy or girl’s number or something?",
        "Bit": "The user sees boxes of Dunkin’ Donuts and Baskin-Robbins on the table, but doesn’t know if they should eat it or not since no one says anything."
    },
    {
        "Segment": "The user takes a part-time job at a convenience store or cafe to make some extra money on the side. And because they want that Employee of the Month (EOM) reward. For a genuine, authentic experience, it sure is a hell of a lot of work.",
        "Bit": "The user is tasked with either welcoming customers and helping them find goods if they are at a convenience store or being a cashier and receiving calls for orders if they are at a cafe."
    },
    {
        "Segment": "The user surprises their older sibling who is in the middle of a date with their new partner. Like every younger sibling, they proceed to third-wheel for the rest of the day. Poor child…",
        "Bit": "The user accompanies their older sibling and partner to a romantic dinner. At the dinner table, the user decides to do something to break the tension."
    },
    {
        "Segment": "The user is about to finish their exchange program, so they try to cook their homestay dinner as a thank you. Time for a solo visit to the grocery store with a recipe they found on YouTube.",
        "Bit": "The user learns the art of the national cuisine and plays with their food."
    }
]

In [ ]:
# Run the game
def run_game():
    intro = cold_open()
    wrapped_intro = textwrap.fill(intro, width = 100) # can remove the textwrap.fill later
    print("\n" + wrapped_intro + "\n")
    time.sleep(2) # new line

    total_score = 0
    num_rounds = len(skits) # new line

    for round in range(num_rounds):
        print("=" * 100) # new line
        print(f"ROUND {round + 1}") # new line
        print("=" * 100 + "\n") # new line

        current_skit = skits[round]

        full_skit = skitzo(current_skit["Segment"])
        wrapped_skit = textwrap.fill(full_skit, width = 100) # Will delete later before connecting to Streamlit
        #print(full_skit)
        print(wrapped_skit) # new line

        player_action_1 = input("\nWhat do you do/say? > ")

        score_1 = get_humor_score(player_action_1)
        total_score += score_1

        reaction_1 = npc(full_skit, player_action_1, score_1)
        wrapped_reaction_1 = textwrap.fill(reaction_1, width = 100)
        #wrapped_reaction = textwrap.fill(reaction, width = 100) # Will delete later before connecting to Streamlit
        #print(f"\n[Evaluator Score: {score}/1]") # May want to delete this later so it doesn't show
        #print(f"{reaction}\n")
        print(f"\n[Evaluator Score: {score_1}/1]") # Again, might delete later
        print(f"{wrapped_reaction_1}\n") # new line

        time.sleep(1)

        full_bit = bit(full_skit, player_action_1, reaction_1, current_skit["Bit"])
        wrapped_bit = textwrap.fill(full_bit, width = 100)
        print(wrapped_bit)

        player_action_2 = input("\nWhat do you do/say? > ")

        score_2 = get_humor_score(player_action_2)
        total_score += score_2

        reaction_2 = npc(full_bit, player_action_2, score_2, wrap = True)
        wrapped_reaction_2 = textwrap.fill(reaction_2, width = 100)
        print(f"\n[Evaluator Score: {score_2}/1]") # might delete later
        print(f"{wrapped_reaction_2}\n")

        time.sleep(2)

    # Define the endings from the branching
    print("=" * 50 + "\nGAME OVER\n" + "=" * 50)
    if total_score == 10:
        print("You did it.")
    elif total_score > 5:
        print("You were aight.")
    else:
        print("You suck.")

In [ ]:
# Start here!
run_game()


[SCENE START]  **INT. THE COMMAND CENTER - NIGHT**  The room is dimly lit, glowing only from the
light of several oversized monitors displaying lines of green scrolling code. **THE LEAD DEVELOPER**
(wearing a hoodie and a headset) sits frantically typing.   **THE DUNGEON MASTER** (wearing a velvet
cape over a "World’s Okayest Boss" t-shirt) paces behind him, clutching a 20-sided die like a stress
ball.  **DUNGEON MASTER** Talk to me, Greg! The players are in the tavern. They’ve been standing
there for twenty minutes. The Rogue is trying to pickpocket a sentient rug!  **LEAD DEVELOPER** I’m
trying, sir! But the text-parser is hanging. Every time someone types "Attack," the game thinks
they’re trying to "Attach" a PDF.   **DUNGEON MASTER** A PDF? We’re in a high-fantasy realm! There
are no Adobe Acrobatics here!   **LEAD DEVELOPER** It gets worse. The Bard tried to seduce the
dragon, but because of a syntax error, he ended up marrying a nearby mailbox. They’re currently
registered at Cr